# DeepBrief.AI - Stage 3: LLM Summarisation & Action Item Extraction

Passes each **full meeting transcript** (all clips concatenated) to Groq.

**Why meeting-level, not clip-level:**  
Individual AMI clips are 2-10 words each - far too short for meaningful summarisation.  
Stage 3 groups all clips by `meeting_id` and concatenates them into one full meeting text  
before calling the LLM. This produces 10 rich outputs instead of 200 mostly-empty ones.

| | |
|---|---|
| **Input** | `data/transcripts.json` (Stage 2) |
| **Output** | `data/stage3_output.json` - one record per meeting |
| **Model** | `openai/gpt-oss-120b` via Groq |
| **Prompts** | 2 per meeting - Summary/Decisions + Action Items |

**Run Stage 2 first.**

## ⚙️ Imports & Configuration

`GROQ_LLM_MODEL` is set to `gpt-oss-120b` — fast, free on Groq, and capable for summarisation. `MAX_TRANSCRIPT_CHARS` truncates very long transcripts to stay within the
model's 8192-token context window.

In [1]:
# ── Standard library ──────────────────────────────────────────────────────────
import json
import os
import re
import time
from pathlib import Path
from typing import Any
#!pip install groq python-dotenv
# ── Third-party ───────────────────────────────────────────────────────────────
from dotenv import load_dotenv
from groq import Groq

# ── Load .env (GROQ_API_KEY must be defined there) ────────────────────────────
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise EnvironmentError("GROQ_API_KEY not found. Add it to your .env file.")

print("Environment loaded.")
print(f"  API key   : {'*' * 8}{GROQ_API_KEY[-4:]}")

Environment loaded.
  API key   : ********IqWm


In [2]:
# ── Model & I/O paths ─────────────────────────────────────────────────────────
MODEL            = "openai/gpt-oss-120b"
INPUT_PATH       = Path("data/transcripts.json")
OUTPUT_PATH      = Path("data/stage3_output.json")
MAX_TOKENS       = 1024        # sufficient for structured JSON responses
TEMPERATURE      = 0.0         # deterministic output reduces hallucination
MAX_RETRIES      = 3           # retry count on transient errors

# ── Rate limit config ─────────────────────────────────────────────────────────
# Groq free tier: 30 requests/min. Each transcript = 2 calls (summary + actions).
# REQUESTS_PER_BATCH caps how many calls are fired before a mandatory 60s pause.
REQUESTS_PER_BATCH = 28        # calls per batch before pausing
BATCH_PAUSE        = 62        # seconds to pause between batches (62 > 60 for safety)
BETWEEN_CALL_SLEEP = 0.5       # small gap between individual calls within a batch

# ── Groq client ───────────────────────────────────────────────────────────────
client = Groq(api_key=GROQ_API_KEY)

print(f"Model              : {MODEL}")
print(f"Input              : {INPUT_PATH}")
print(f"Output             : {OUTPUT_PATH}")
print(f"Temperature        : {TEMPERATURE}")
print(f"Requests per batch : {REQUESTS_PER_BATCH}")
print(f"Pause between batch: {BATCH_PAUSE}s")


Model              : openai/gpt-oss-120b
Input              : data\transcripts.json
Output             : data\stage3_output.json
Temperature        : 0.0
Requests per batch : 28
Pause between batch: 62s


## 📝 Prompt Templates


Two separate prompts are used to keep each LLM call focused and reduce confusion:
- **Prompt 1**: Extracts high-level summary, key discussion points, and decisions.
- **Prompt 2**: Extracts concrete action items with owner and deadline.

Design choices to reduce hallucination:
- `temperature=0.0` — no randomness in generation
- Explicit JSON schema embedded in the prompt
- "Only use information from the transcript" instruction
- `null` explicitly allowed for missing fields
- "Do not invent" instructions on every field that could be fabricated

In [3]:
SYSTEM_PROMPT = (
    "You are a precise meeting analyst. "
    "You extract structured information ONLY from the transcript provided. "
    "You never invent, infer, or assume information that is not explicitly stated. "
    "CRITICAL: Respond with a single raw JSON object and absolutely nothing else. "
    "No prose before the JSON. No prose after the JSON. No markdown. No code fences. "
    "The very first character of your response must be { and the very last must be }."
)

In [4]:
def build_summary_prompt(transcript: str) -> str:
    """Prompt 1: Extract summary, key discussion points, and decisions."""
    return (
        "Extract structured information from the meeting transcript below.\n\n"
        "Respond with this JSON object and nothing else — no explanation, no markdown:\n"
        "{\n"
        "  \"summary\": \"2-4 sentence overview of what was discussed\",\n"
        "  \"key_points\": [\"topic or issue raised\", \"another topic\"],\n"
        "  \"decisions\": [\"explicit decision made\"]\n"
        "}\n\n"
        "Rules:\n"
        "- Use ONLY information stated in the transcript.\n"
        "- If no decisions were made, use an empty list: []\n"
        "- If the transcript is too short or unclear, still return the JSON with empty fields.\n"
        "- First character of response: {   Last character: }\n\n"
        f"TRANSCRIPT:\n---\n{transcript}\n---"
    )

print("Summary prompt defined.")

Summary prompt defined.


In [5]:
def build_actions_prompt(transcript: str) -> str:
    """Prompt 2: Extract action items with task, owner, and deadline."""
    return (
        "Extract action items from the meeting transcript below.\n\n"
        "Respond with this JSON object and nothing else — no explanation, no markdown:\n"
        "{\n"
        "  \"action_items\": [\n"
        "    {\n"
        "      \"task\": \"specific task to be done\",\n"
        "      \"owner\": \"person responsible or null\",\n"
        "      \"deadline\": \"deadline or timeframe or null\"\n"
        "    }\n"
        "  ]\n"
        "}\n\n"
        "Rules:\n"
        "- Extract ONLY tasks explicitly mentioned in the transcript.\n"
        "- Use JSON null (not the string null) when owner or deadline is not stated.\n"
        "- If no action items exist, return: {\"action_items\": []}\n"
        "- First character of response: {   Last character: }\n\n"
        f"TRANSCRIPT:\n---\n{transcript}\n---"
    )

print("Actions prompt defined.")

Actions prompt defined.


## 🔧 LLM Call & Parse Functions

`call_llm()` wraps the Groq chat completion with a single retry on failure — covers the
most common transient error (rate limit spike). 

`safe_parse_json()` handles JSON parse failures gracefully by stripping markdown fences
the model sometimes adds despite instructions, then returning an empty list if parsing still fails.

In [6]:
def call_llm(user_prompt: str, retries: int = MAX_RETRIES) -> str:
    """
    Send a prompt to the Groq LLM and return the raw text response.

    Retries on transient errors with exponential backoff.
    Returns an empty string on permanent failure.
    """
    for attempt in range(1, retries + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_prompt},
                ],
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
            )
            return response.choices[0].message.content.strip()

        except Exception as exc:
            wait = 2 ** attempt
            print(f"    [attempt {attempt}/{retries}] Error: {exc}. Retrying in {wait}s...")
            time.sleep(wait)

    print("    [ERROR] All retries exhausted. Returning empty string.")
    return ""

In [7]:
def safe_parse_json(raw: str, fallback: dict) -> dict:
    """
    Robustly extract a JSON object from an LLM response using 4 strategies:
      1. Direct parse       — whole response is valid JSON (ideal)
      2. Fence strip        — strip ```json ... ``` then parse
      3. Brace extract      — find first { and last } and parse between them
      4. Regex largest block — scan for the largest {...} block in the response
    Returns fallback only if all four strategies fail.
    """
    if not raw or not raw.strip():
        return fallback

    text = raw.strip()

    # Strategy 1: direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Strategy 2: strip markdown fences
    cleaned = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # Strategy 3: extract between first { and last }
    start = text.find("{")
    end   = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(text[start:end + 1])
        except json.JSONDecodeError:
            pass

    # Strategy 4: find the largest valid {...} block
    for match in sorted(re.finditer(r"\{.*?\}", text, re.DOTALL),
                        key=lambda m: len(m.group()), reverse=True):
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            continue

    print(f"    [WARN] All JSON extraction strategies failed.")
    print(f"    [WARN] Raw response (first 300 chars): {raw[:300]}")
    return fallback

In [8]:
# Fallback structures returned when parsing fails — keeps output schema consistent
SUMMARY_FALLBACK = {"summary": "", "key_points": [], "decisions": []}
ACTIONS_FALLBACK = {"action_items": []}

print("Utilities defined.")

Utilities defined.


## Load & Group Transcripts from Stage 2

Loads `transcripts.json`, groups all clips by meeting ID, and concatenates transcripts  
into one full meeting text per meeting - the same grouping Stage 4 uses.  
`get_meeting_id()` extracts the meeting ID from a clip ID like `EN2001a_clip03`.

In [9]:
import re
from collections import defaultdict

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_PATH}\n"
        "Ensure Stage 2 has completed and saved transcripts.json."
    )

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    clips: list[dict] = json.load(f)

for clip in clips:
    if "id" not in clip and "sample_id" in clip:
        clip["id"] = clip["sample_id"]

def get_meeting_id(clip_id: str) -> str:
    # Extract meeting ID from clip ID e.g. EN2001a_clip03 -> EN2001a
    match = re.match(r'^([A-Z]{2}\d{4}[a-z])_clip\d+$', clip_id)
    return match.group(1) if match else clip_id

meeting_groups: dict = defaultdict(lambda: {"clip_ids": [], "transcript_parts": []})

for clip in clips:
    mid  = get_meeting_id(clip["id"])
    text = clip.get("transcript", "").strip()
    meeting_groups[mid]["clip_ids"].append(clip["id"])
    if text:
        meeting_groups[mid]["transcript_parts"].append(text)

meetings = []
for mid, data in meeting_groups.items():
    full_text = " ".join(data["transcript_parts"])
    meetings.append({
        "meeting_id":      mid,
        "clip_ids":        data["clip_ids"],
        "full_transcript": full_text,
        "word_count":      len(full_text.split()),
    })

print(f"Loaded {len(clips)} clips -> grouped into {len(meetings)} meetings\n")
print(f"{'Meeting ID':<15} {'Clips':>6} {'Words':>8}  Preview")
print(f"{'-'*15} {'-'*6} {'-'*8}  {'-'*50}")
for m in meetings:
    print(f"  {m['meeting_id']:<13} {len(m['clip_ids']):>6} {m['word_count']:>8}  {m['full_transcript'][:50]}...")

Loaded 200 clips -> grouped into 10 meetings

Meeting ID       Clips    Words  Preview
--------------- ------ --------  --------------------------------------------------
  EN2001a           20      266  if you SS agent they have this big warning about d...
  EN2001b           20      255  This is how much time you spend just getting the r...
  EN2001d           20      237  It's ready to get started. And this is summarized ...
  EN2001e           20      252  but it's probably not a simple thing. The interest...
  EN2003a           20      219  Because, I mean, would six hours a day be pushing ...
  EN2004a           20      191  So would you have been tempted to put an asterisk ...
  EN2005a           20      243  happening in the meeting. but that was not well pi...
  EN2006a           20      169  But it's actually Merlin and all the international...
  EN2006b           20      141  Yes, assuming it's talking about starburst galaxie...
  EN2009b           20      177  So everyone s

## Main Processing Loop

Calls the LLM twice per **meeting** (not per clip):  
1. Summary call - generates structured summary, key points, and decisions  
2. Action items call - extracts concrete tasks, owners, and deadlines  

10 meetings x 2 calls = 20 total API calls. Rate limiting is not a concern here  
but the batch pausing logic is kept for safety.  
Results are keyed by `meeting_id` so Stage 4 can look them up directly.

In [10]:
results: list[dict[str, Any]] = []
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

total         = len(meetings)
request_count = 0

print(f"Processing {total} meetings (2 Groq calls each)...\n")
print("=" * 60)

for idx, meeting in enumerate(meetings, start=1):
    meeting_id = meeting["meeting_id"]
    transcript = meeting["full_transcript"]

    print(f"[{idx}/{total}] {meeting_id}  |  {meeting['word_count']} words  |  {len(meeting['clip_ids'])} clips")

    if not transcript:
        print(f"  [SKIP] No transcript text for this meeting.")
        results.append({
            "id":            meeting_id,
            "meeting_id":    meeting_id,
            "clip_ids":      meeting["clip_ids"],
            "summary":       SUMMARY_FALLBACK,
            "actions":       ACTIONS_FALLBACK,
        })
        with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        continue

    if request_count + 2 > REQUESTS_PER_BATCH:
        print(f"  [RATE LIMIT] {request_count} requests fired - pausing {BATCH_PAUSE}s...")
        for remaining in range(BATCH_PAUSE, 0, -10):
            print(f"    Resuming in {remaining}s...", end="\r", flush=True)
            time.sleep(min(10, remaining))
        print(f"  [RATE LIMIT] Resuming.                              ")
        request_count = 0

    # Summary call
    print(f"  -> Summarising...", end=" ", flush=True)
    raw_summary  = call_llm(build_summary_prompt(transcript))
    summary_data = safe_parse_json(raw_summary, SUMMARY_FALLBACK)
    request_count += 1
    time.sleep(BETWEEN_CALL_SLEEP)
    print(f"done  ({len(summary_data.get('key_points', []))} key points, {len(summary_data.get('decisions', []))} decisions)")

    # Action items call
    print(f"  -> Extracting action items...", end=" ", flush=True)
    raw_actions  = call_llm(build_actions_prompt(transcript))
    actions_data = safe_parse_json(raw_actions, ACTIONS_FALLBACK)
    request_count += 1
    time.sleep(BETWEEN_CALL_SLEEP)
    print(f"done  ({len(actions_data.get('action_items', []))} items)")

    record = {
        "id":            meeting_id,
        "meeting_id":    meeting_id,
        "clip_ids":      meeting["clip_ids"],
        "word_count":    meeting["word_count"],
        "full_transcript": transcript,
        "summary":       summary_data,
        "actions":       actions_data,
    }
    results.append(record)

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print()

print("=" * 60)
print(f"Done. {len(results)} meetings processed -> {OUTPUT_PATH}")

Processing 10 meetings (2 Groq calls each)...

[1/10] EN2001a  |  266 words  |  20 clips
  -> Summarising... done  (10 key points, 0 decisions)
  -> Extracting action items... done  (2 items)

[2/10] EN2001b  |  255 words  |  20 clips
  -> Summarising... done  (6 key points, 1 decisions)
  -> Extracting action items... done  (0 items)

[3/10] EN2001d  |  237 words  |  20 clips
  -> Summarising... done  (4 key points, 0 decisions)
  -> Extracting action items... done  (3 items)

[4/10] EN2001e  |  252 words  |  20 clips
  -> Summarising... done  (7 key points, 0 decisions)
  -> Extracting action items... done  (1 items)

[5/10] EN2003a  |  219 words  |  20 clips
  -> Summarising... done  (5 key points, 0 decisions)
  -> Extracting action items... done  (1 items)

[6/10] EN2004a  |  191 words  |  20 clips
  -> Summarising... done  (5 key points, 0 decisions)
  -> Extracting action items... done  (3 items)

[7/10] EN2005a  |  243 words  |  20 clips
  -> Summarising... done  (5 key points,

## 📋Save Output

In [11]:
# Ensure output directory exists
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Output saved to: {OUTPUT_PATH}")
print(f"Records written: {len(results)}")

Output saved to: data\stage3_output.json
Records written: 10


## 📊 — Validation & Quick Preview

In [13]:
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    verification = json.load(f)

total_meetings       = len(verification)
summary_ok           = sum(1 for e in verification if e.get("summary", {}).get("summary"))
total_key_points     = sum(len(e.get("summary", {}).get("key_points", [])) for e in verification)
total_decisions      = sum(len(e.get("summary", {}).get("decisions", [])) for e in verification)
total_actions        = sum(len(e.get("actions", {}).get("action_items", [])) for e in verification)
meetings_with_actions = sum(1 for e in verification if e.get("actions", {}).get("action_items"))

print("=" * 60)
print("Stage 3 - Output Quality Summary (meeting-level)")
print("=" * 60)
print(f"  Meetings processed         : {total_meetings}")
print(f"  Summaries generated        : {summary_ok}/{total_meetings}")
print(f"  Total key points           : {total_key_points}")
print(f"  Total decisions            : {total_decisions}")
print(f"  Meetings with action items : {meetings_with_actions}/{total_meetings}")
print(f"  Total action items found   : {total_actions}")
avg = f"{total_actions / total_meetings:.1f}" if total_meetings else "N/A"
print(f"  Avg action items / meeting : {avg}")
print()
print(f"{'Meeting':<12} {'Words':>6} {'Key Pts':>8} {'Decisions':>10} {'Actions':>8}  Summary preview")
print(f"{'-'*12} {'-'*6} {'-'*8} {'-'*10} {'-'*8}  {'-'*40}")
for e in verification:
    s = e.get("summary", {})
    a = e.get("actions", {})
    preview = s.get("summary", "")[:50]
    print(f"  {e['meeting_id']:<10} {e.get('word_count',0):>6} "
          f"{len(s.get('key_points',[])):>8} "
          f"{len(s.get('decisions',[])):>10} "
          f"{len(a.get('action_items',[])):>8}  {preview}...")
print()
print("Stage 3 complete. Proceed to Stage 4.")

Stage 3 - Output Quality Summary (meeting-level)
  Meetings processed         : 10
  Summaries generated        : 10/10
  Total key points           : 62
  Total decisions            : 2
  Meetings with action items : 6/10
  Total action items found   : 11
  Avg action items / meeting : 1.1

Meeting       Words  Key Pts  Decisions  Actions  Summary preview
------------ ------ -------- ---------- --------  ----------------------------------------
  EN2001a       266       10          0        2  The participants discussed technical consideration...
  EN2001b       255        6          1        0  The speakers discussed the time required to set up...
  EN2001d       237        4          0        3  The participants discussed meeting after 3 pm to f...
  EN2001e       252        7          0        1  The participants talked about technical challenges...
  EN2003a       219        5          0        1  The participants debated the appropriate length of...
  EN2004a       191        5  